## 🧱 NETWORK ARCHITECTURE OVERVIEW

### Layers

```python
Input Layer      →     Hidden Layer      →     Output Layer
 (X, shape: NxD)       (a_1, shape: NxH)       (a_2, shape: Nx1)
```

* `N` = number of samples in batch
* `D` = input features
* `H` = number of hidden units

---

## 🔵 FORWARD PASS VISUALIZED

```python
Z_1 = X @ w_1 + b_1      # shape: (N, H)
a_1 = sigmoid(Z_1)       # shape: (N, H)

Z_2 = a_1 @ w_2 + b_2    # shape: (N, 1)
a_2 = sigmoid(Z_2)       # shape: (N, 1)
```

### Visualization:

```python
      X (N x D)
         │
         ▼
     Linear Layer (w_1: D x H, b_1: 1 x H)
         │
         ▼
    Z_1 = X @ w_1 + b_1   → shape: N x H
         │
         ▼
    a_1 = sigmoid(Z_1)
         │
         ▼
     Linear Layer (w_2: H x 1, b_2: 1 x 1)
         │
         ▼
    Z_2 = a_1 @ w_2 + b_2 → shape: N x 1
         │
         ▼
    a_2 = sigmoid(Z_2) → final output probs (0–1)
```

---

## 🔴 BACKWARD PASS VISUALIZED

```python
delta_2 = a_2 - y
delta_1 = (delta_2 @ w_2.T) * a_1 * (1 - a_1)
```

### Compute Gradients

```python
       Loss
         ▲
         │ dLoss/da_2
         │
         ◄── delta_2 = (a_2 - y)
                     │
                     ▼
         Gradient for w_2: a_1.T @ delta_2
                     │
         ◄───────────┘

Hidden Layer:
         ▲
         │
         ◄── delta_1 = (delta_2 @ w_2.T) * sigmoid_deriv(a_1)
                     │
                     ▼
         Gradient for w_1: X.T @ delta_1
```

### Visualization:

```python
         Loss (Binary Cross Entropy)
                    ▲
                    │
            delta_2 = a_2 - y
                    ▲
                    │
         dL/dw_2 = a_1.T @ delta_2
         dL/db_2 = sum(delta_2)

Backprop to hidden layer:
     delta_1 = (delta_2 @ w_2.T) * sigmoid'(Z_1)
                    ▲
                    │
         dL/dw_1 = X.T @ delta_1
         dL/db_1 = sum(delta_1)
```

---

## 🔁 FULL STEP-BY-STEP BACKWARD PASS IN PLAIN LANGUAGE

### 1. Calculate error in prediction

```python
delta_2 = a_2 - y
```

> *"How far are we from the correct answer?"*

---

### 2. Backpropagate the error to hidden layer

```python
delta_1 = (delta_2 @ w_2.T) * a_1 * (1 - a_1)
```

> *"How much did each hidden neuron contribute to the error?"*

---

### 3. Update weights to reduce error

```python
w_2 -= lr * a_1.T @ delta_2
w_1 -= lr * X.T @ delta_1
```

> *"Move weights in the direction that reduces the error."*

---

### 4. Update biases similarly

```python
b_2 -= lr * sum(delta_2)
b_1 -= lr * sum(delta_1)
```

---

## 📊 BONUS: ASCII-style Summary Chart

```python
        INPUT         HIDDEN         OUTPUT
    ┌────────────┐  ┌───────────┐  ┌────────────┐
    │    X       │→ │   Z_1     │→ │   Z_2      │
    │  (N x D)   │  │ (N x H)   │  │  (N x 1)   │
    └────────────┘  └────┬──────┘  └────┬───────┘
                          │             │
                      sigmoid        sigmoid
                          │             │
                          ▼             ▼
                     ┌───────────┐  ┌────────────┐
                     │   a_1     │  │   a_2      │
                     │ (N x H)   │  │  (N x 1)   │
                     └───────────┘  └────┬───────┘
                                        ▼
                                  Binary Cross Entropy
                                        │
                                        ▼
                                Loss and Backward Pass
```

### 🧱 Model:

```python
Input (2) → Hidden (2) → Output (1)
```

## 🧮 Step 1: Forward Pass (Sample Data)

Let’s define:

```python
X = [[1, 2]]          # Input (1x2)
y = [[1]]             # True label (1x1)

w_1 = [[0.1, 0.3],    # Weights from input to hidden
       [0.2, 0.4]]    # shape: 2x2

b_1 = [[0.0, 0.0]]    # Biases for hidden layer

w_2 = [[0.5],
       [0.6]]         # Weights from hidden to output (2x1)

b_2 = [[0.0]]         # Output bias
```

---

## 🚀 Forward Pass

### 🔹 Hidden Layer

```python
Z_1 = X @ w_1 + b_1
    = [[1, 2]] @ [[0.1, 0.3],
                  [0.2, 0.4]]
    = [[1×0.1 + 2×0.2, 1×0.3 + 2×0.4]]
    = [[0.5, 1.1]]

a_1 = sigmoid(Z_1)
    = sigmoid([0.5, 1.1]) ≈ [0.622, 0.750]
```

### 🔹 Output Layer

```python
Z_2 = a_1 @ w_2 + b_2
    = [[0.622, 0.750]] @ [[0.5],
                          [0.6]]
    = [0.622×0.5 + 0.750×0.6] = [0.311 + 0.45] = [0.761]

a_2 = sigmoid(0.761) ≈ 0.681
```

✅ Output prediction: **0.681**
✅ True label: **1**
✅ Time to backprop.

---

## 🔁 Backward Pass — Line by Line

### 🔴 Step 1: Compute output error

```python
delta_2 = a_2 - y = 0.681 - 1 = -0.319
```

> *"We overshot the correct label by \~0.32"*

---

### 🔴 Step 2: Backprop to hidden layer

We need:

```python
delta_1 = (delta_2 @ w_2.T) * a_1 * (1 - a_1)
```

Break it down:

#### a) delta\_2 @ w\_2.T:

```python
w_2.T = [[0.5, 0.6]]         # shape: (1,2)
delta_2 @ w_2.T = -0.319 @ [[0.5, 0.6]] = [-0.1595, -0.1914]
```

#### b) sigmoid\_derivative(a\_1):

```python
a_1 = [0.622, 0.750]
sigmoid'(a_1) = a_1 * (1 - a_1) = [0.622×(1-0.622), 0.75×(1-0.75)]
              ≈ [0.235, 0.188]
```

#### c) delta\_1:

```python
delta_1 = [-0.1595, -0.1914] * [0.235, 0.188]
        ≈ [-0.0375, -0.0360]
```

---

### 🔴 Step 3: Compute Gradients and Update Weights

#### Gradients for `w_2`:

```python
a_1.T = [[0.622],
         [0.750]]

grad_w2 = a_1.T @ delta_2
        = [[0.622×-0.319], [0.750×-0.319]] ≈ [[-0.1984], [-0.2393]]
```

#### Gradients for `w_1`:

```python
X.T = [[1],
       [2]]

grad_w1 = X.T @ delta_1
        = [[1×-0.0375, 1×-0.0360],
           2×-0.0375, 2×-0.0360]] = [[-0.0375, -0.0360],
                                     [-0.0750, -0.0720]]
```

#### Gradients for biases:

```python
grad_b2 = delta_2 = -0.319
grad_b1 = delta_1 = [-0.0375, -0.0360]
```

---

### ✅ Update Weights (Assume lr = 0.1)

```python
w_2 -= 0.1 * grad_w2 → w_2 += 0.0198, 0.0239
w_1 -= 0.1 * grad_w1 → w_1 += 0.00375, etc.
b_2 -= 0.1 * grad_b2 → b_2 += 0.0319
b_1 -= 0.1 * delta_1 → b_1 += [0.00375, 0.0036]
```

---

## 🧠 Final Visualization Summary

```python
INPUT:         X = [[1, 2]]
             w_1 = 2x2, b_1 = 1x2
             w_2 = 2x1, b_2 = 1x1

FORWARD:
    Z_1 = X @ w_1 + b_1 → (1x2)
    a_1 = sigmoid(Z_1)

    Z_2 = a_1 @ w_2 + b_2 → (1x1)
    a_2 = sigmoid(Z_2)

BACKWARD:
    delta_2 = a_2 - y
    delta_1 = (delta_2 @ w_2.T) * sigmoid'(Z_1)

    grad_w2 = a_1.T @ delta_2
    grad_w1 = X.T @ delta_1

    Update weights: w_1, w_2, b_1, b_2
```

---

## 🧰 TL;DR Table

| Variable  | Shape  | Meaning                        |
| --------- | ------ | ------------------------------ |
| `X`       | (1, 2) | Input features                 |
| `w_1`     | (2, 2) | Input → Hidden weights         |
| `b_1`     | (1, 2) | Hidden biases                  |
| `Z_1`     | (1, 2) | Pre-activation of hidden layer |
| `a_1`     | (1, 2) | Hidden activations             |
| `w_2`     | (2, 1) | Hidden → Output weights        |
| `Z_2`     | (1, 1) | Pre-activation of output layer |
| `a_2`     | (1, 1) | Prediction (output activation) |
| `delta_2` | (1, 1) | Gradient at output             |
| `delta_1` | (1, 2) | Gradient at hidden layer       |

In [1]:
import numpy as np
from tqdm import tqdm

In [2]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def binary_cross_entropy(y_pred, y_true):
    y_pred = np.clip(y_pred, 1e-9, 1 - 1e-9)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [51]:
class NeuralNetwork():
    def __init__(self, X : np.ndarray, y : np.ndarray, input_size : int, hidden_size : int, lr=0.1):
        self.X = X
        self.y = y

        self.w_1 = np.random.randn(input_size, hidden_size)
        self.b_1 = np.zeros((1, hidden_size))
        self.w_2 = np.random.randn(hidden_size, 1)
        self.b_2 = np.zeros((1,1))
        self.lr = lr

    def forward(self):
        self.Z_1 = self.X @ self.w_1 + self.b_1
        self.a_1 = sigmoid(self.Z_1)
        self.Z_2 = self.a_1 @ self.w_2 + self.b_2
        self.a_2 = sigmoid(self.Z_2)
        
        return self.a_2

    def backward(self):

        delta_2 = (self.a_2 - self.y) * self.a_2 * (1 - self.a_2)
        delta_1 = (delta_2 @ self.w_2.T) * self.a_1 * (1 - self.a_1) 
        
        self.w_2 -= self.lr * self.a_1.T @ delta_2
        self.w_1 -= self.lr * self.X.T @ delta_1

        self.b_2 -= self.lr * np.sum(delta_2, axis=0, keepdims=True)
        self.b_1 -= self.lr * np.sum(delta_1, axis=0, keepdims=True)



    def train(self, epoch: int):
        for i in range(epoch):
            preds = self.forward()
            self.backward()

            if i % 100 == 0 or i == epoch - 1:
                loss = binary_cross_entropy(preds, self.y)
                acc = np.mean((preds > 0.5).astype(int) == self.y)
                print(f"Epoch {i}: Loss = {loss:.4f}, Accuracy = {acc:.2f}")



    def predict(self, X):
        Z1 = X @ self.w_1 + self.b_1
        a1 = sigmoid(Z1)

        Z2 = a1 @ self.w_2 + self.b_2
        a2 = sigmoid(Z2)

        # Return class label: 0 or 1
        return (a2 > 0.5).astype(int)
    
    def evaluate(self, X, y):
        preds = self.predict(X)
        accuracy = np.mean(preds == y)
        return accuracy

In [54]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([[0], [1], [1], [0]])  # shape = (4, 1)

model = NeuralNetwork(X, y,input_size=2, hidden_size=4, lr=0.1)
model.train(20)

Epoch 0: Loss = 0.7407, Accuracy = 0.50
Epoch 19: Loss = 0.7100, Accuracy = 0.75
